![image_1787419385294.png](./image_1787419385294.png "image_1787419385294.png")

# Contexto
En el sector asegurador, la detección temprana de posibles casos de fraude permite focalizar esfuerzos de investigación, proteger la sostenibilidad técnica del portafolio y mejorar la eficiencia operativa de los equipos de siniestros, auditoría y gestión del riesgo. Esta prueba busca evaluar la capacidad del candidato para transformar una necesidad de negocio en un problema analítico abordable, construir un modelo predictivo con datos reales o simulados, interpretar sus resultados y comunicar sus hallazgos de forma clara para audiencias técnicas y de negocio.


#Reto técnico
Construir una solución analítica para estimar la probabilidad de que un registro, siniestro, reclamación, transacción o caso del negocio asegurador corresponda a un posible fraude. El candidato deberá trabajar con una única matriz de datos suministrada por la compañía, cuya variable objetivo es FraudeS /N, y desarrollar un flujo completo de análisis y modelado, desde la exploración inicial hasta la sustentación de resultados.
Se dará puntos adicionales si la solución es desarrollada en Databricks Free Edition, en caso contrario deberá ser implementada en Python, Incluyendo Git para control de versiones con un repositorio organizado.

#Configuración del repositorio
Es importante tener un versionamiento del proyecto por lo que se vincula este notebook a un repositorio previamente creado y sincronizado con databricks

#Instalación de paquetes

In [0]:
#instalamos la librerías
#para lectura de datos
%pip install -q openpyxl
#para mixed nulls
%pip install -q deepchecks --upgrade
#para estadistica descriptiva
%pip install -q "pathspec<0.12"
%pip install -q scikit-build-core cmake ninja pybind11
%pip install -q --no-build-isolation "phik==0.12.5"
%pip install -q ydata_profiling
#para catboost
%pip install -q catboost

#Reinicio del entorno

In [0]:
%restart_python

#Importación de librerías

In [0]:
#Manejo de datos
import pandas as pd
import numpy as np

if not hasattr(np, "Inf"):
    np.Inf = np.inf

from sklearn.preprocessing import LabelEncoder
#librerías gráficas
import seaborn as sns
import matplotlib.pyplot as plt

#Reconocimiento de nulos
from deepchecks.tabular.checks import MixedNulls
#Validación cruzada
from sklearn.model_selection import KFold
#Modelación
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from joblib import dump
#métricas de evaluación
from sklearn.metrics import accuracy_score
from sklearn import metrics
#hiperparametrización
from sklearn.model_selection import GridSearchCV
#Correlacion
from scipy.stats import chi2_contingency
#Descripcion estadistica
from ydata_profiling import ProfileReport

#Lectura de los datos
Hacemos la lectura de los datos que ya previamente fueron subidos al catalog de databricks, adicionalmente miramos unos cuantos registros para entender mejor la estructura de los datos y el formato de las variables

In [0]:
Ruta_base = "/Volumes/workspace/prueba_tecnica/muestra_base_fraude/Muestra_Base_fraude.xlsx"
Base = pd.read_excel(Ruta_base) 
Base.head(5)

*   Contamos con una base de **12.776 filas** y **40 columnas** , esto puede cambiar a medida de que hagamos transformaciones y limpieza.

*   Podemos apreciar el **tipo de dato** y **cantidad de no nulos** en cada variable, esto último se debe analizar junto con **conocimiento de negocio** ya que en el caso de algunas variables puede ser **normal** el tener muchos nulos
*   Adicionalmente nos podemos dar una idea del **nivel de completitud** de cada variable, (esto no quere decir que sea el escenario final de completitud , ya que los varores **nulos** pueden estar en **diferentes formatos**, no todos necesariamente detectados por la función)

In [0]:
Base.info()

#Detección de registros duplicados
Primero inspeccionamos la base virgen en busca de registros repetidos y al parecer tenemos 329 reclamos Duplicados. (Puede que mas adelante cuando tenga mas entendimiento sobre la base pueda hacer combinaciones buscando otro tipo de duplicidades) 

In [0]:
##Detección de registros duplicados
Base.duplicated().value_counts()

Hacemos una primera limpieza de **registros duplicados** los cuales pueden ensuciar nuestros futuros modelos , nos quedamos con **12447 registros unicos** , con eso nos aseguramos que cada registro es una reclamacion unica.

In [0]:
Base_sin_duplicados = Base.drop_duplicates().reset_index(drop=True)

#Entendimiento de los datos
* Segun se puede ver a simple vista esta base de datos representa un conjunto de reclamaciones de seguros de vida (rentas, invalidez, incapacidades) hechas por clientes en diferentes ventanas de tiempo , cada registro trae todo el ciclo del siniestro "desde la vigencia de la póliza hasta el cierre" junto con la etiqueta de si fue detectado como fraude o no

* Con el fin de conocer mas a fondo la naturaleza de la base y asi poder descartar todas las variables que por definición son irrelevantes para la construccion del modelo , se crea un glosario con el entendimiento de cada variable

### Producto y canal de venta

- **Ramo**: código del ramo del seguro.
- **Ramo_Desc**: descripción del ramo (debe ser homóloga a *Ramo*).
- **Nombre_plan**: producto específico al que pertenece la reclamación.
- **Codigo_Canal_Comercial_Op** / **Nombre_Canal_Comercial**: código y nombre del canal por el que se vendió el seguro.
- **Amparo_Desc**: nombre del amparo (cobertura) afectado por la reclamación.

### Póliza y asegurado

- **Fecha_Primera_Vigencia_Cert**: fecha de primera vigencia del certificado individual (me interesa mas esta fecha por que es mas exacta que la master).
- **fecha_primera_vigencia_pol**: fecha de primera vigencia de la póliza máster. En seguros individuales debería coincidir con la del certificado.
- **IDENTIFICACION_asegurado**: número de identificación del asegurado.
- **SEXO_asegurado**: género del asegurado.
- **edad_ingreso_asegurado** / **edad_actual_asegurado**: edad al vincularse a la compañía vs. edad al momento de la reclamación (gran potencial para definir antiguedad).
- **vigencia_poliza**: número de vigencias/renovaciones de la póliza; funciona como proxy de antigüedad del cliente.
- **vigencia_certificado**: similar a *vigencia_poliza* pero a nivel de certificado.
- **FEXPEDICION**: todo indica que es la fecha de expedición de la póliza/certificado, no del reclamo. Coincide de cerca con las fechas de primera vigencia, y en 91% de los casos es anterior a *FSINIESTRO* — la póliza se expide antes de que ocurra el siniestro, como debería ser.

### Estructura comercial

- **CODSUC** / **SUCURSAL**: código y nombre de la oficina.
- **REGIONAL**: regional a la que pertenece la oficina.
- **AGENTE** / **CODAG**: nombre e identificador del agente o asociación que vendió el seguro.

### El siniestro en sí

- **CAUSASTRO** / **DESCAUSA**: id y nombre de la causa de la reclamación.
- **DIAGNOSTICO**: diagnóstico médico asociado.
- **FSINIESTRO**: fecha en que ocurrió el siniestro.
- **F_Notificacion**: fecha en que se notificó el siniestro a la compañía.
- **Fecha_Recepcion**: fecha de recepción formal de la reclamación.
- **Fecha_Apertura**: fecha de apertura del caso.
- **Fecha_Primer_Cierre_Siniestro**: fecha del primer cierre. 
- **Ind_Tipo_Atencion**: canal/modalidad de atención (interna vs. externa).
- **Ind_Pago_Automatico**: si el pago se hizo de forma automática (S/N).

### Montos y estado

- **Sum(Valor_Reservas_Inicial)**: reserva inicial constituida.
- **Sum(Valor_Reservas)**: reserva final/actual.
- **Sum(Valor_Pagos)**: valor efectivamente pagado.
- **estado**: estado actual de la reclamación.
- **Cobertura**: cobertura afectada — se cruza directamente con *Amparo_Desc*.
- **Tipo apertura**: medio por el que se atendió la reclamación, muy relacionada con *Ind_Tipo_Atencion*.

### Variable objetivo y reporte

- **Fraude S /N**: variable objetivo — si la reclamación fue determinada como fraude.
- **Periodo Reporte**: mes del reporte.
- **Fecha de reporte**: fecha completa del reporte del fraude.
- **Año**: año del reporte.

# Eliminacion de variables irrelevantes por definición

- **Ramo**: como id no tiene ningun valor descriptivo, ademas ya existe su version en texto llamada Ramo_Desc.
- **Codigo_Canal_Comercial_Op**: igual que Ramo, no aporta nada como codigo y tiene su homologo descriptivo en Nombre_Canal_Comercial.
- **CODSUC**: el id en si no tiene valor descriptivo, ya existe su homologo SUCURSAL con el nombre de la oficina.
- **CODAG**: comparte la misma descripcion que AGENTE, asi que se queda solo este ultimo.
- **CAUSASTRO**: es un id sin valor descriptivo, su homologo con descripcion es DESCAUSA.

 NOTA: a pesar que la variable IDENTIFICACION_asegurado tiene la misma naturaleza que estas variables , aun no la elimino por que a partir de ella puedo crear otras variables

Eliminamos las variables anteriormente mencionadas, quedamos con 35 de las 40 variables originales

In [0]:
columnas_a_eliminar = ['Ramo', 'Codigo_Canal_Comercial_Op', 'CODSUC', 'CODAG', 'CAUSASTRO']

Base_sin_duplicados = Base_sin_duplicados.drop(columns=columnas_a_eliminar)

print(f"Columnas eliminadas: {columnas_a_eliminar}")
print(f"Dimensiones actuales: {Base_sin_duplicados.shape}")

# Eliminacion de variables redundantes por definición

- **fecha_primera_vigencia_pol**: comparte gran parte de sus valores con Fecha_Primera_Vigencia_Cert (un 68% de sus valores). Me quedo con esta ultima porque da mayor detalle del cliente al ser la vigencia del certificado y no de la poliza master.
- **vigencia_poliza**: comparte varios valores con vigencia_certificado (en un 76%), y al igual que en el caso anterior prefiero quedarme con la version de certificado.
- **F_Notificacion**, **Fecha_Recepcion**, **Fecha_Apertura**: estas tres fechas son casi identicas entre si. Me quedo con **F_Notificacion** porque el nombre es mas diciente y hace referencia directa a la fecha de aviso del siniestro.
- **Periodo Reporte**, **Fecha de reporte**, **Año**: estas tres hacen referencia a la misma fecha en distintos niveles de detalle, asi que me quedo unicamente con **Fecha de reporte** por ser la mas completa.

A continuacion mostramos las coincidencias entre variables mencionadas en el texto anteior, que evidencian su redundancia

In [0]:
coincidencia_vigencias = (Base['Fecha_Primera_Vigencia_Cert'] == Base['fecha_primera_vigencia_pol']).mean()

print(f"Porcentaje de coincidencia exacta entre ambas fechas: {coincidencia_vigencias:.2%}")

In [0]:
coincidencia_vigencia_num = (Base['vigencia_poliza'] == Base['vigencia_certificado']).mean()

print(f"Porcentaje de coincidencia exacta entre ambas vigencias: {coincidencia_vigencia_num:.2%}")

In [0]:
coincidencia_notif_recep = (Base['F_Notificacion'] == Base['Fecha_Recepcion']).mean()
coincidencia_notif_apert = (Base['F_Notificacion'] == Base['Fecha_Apertura']).mean()
coincidencia_recep_apert = (Base['Fecha_Recepcion'] == Base['Fecha_Apertura']).mean()
coincidencia_las_tres = ((Base['F_Notificacion'] == Base['Fecha_Recepcion']) & 
                          (Base['Fecha_Recepcion'] == Base['Fecha_Apertura'])).mean()

print(f"F_Notificacion == Fecha_Recepcion: {coincidencia_notif_recep:.2%}")
print(f"F_Notificacion == Fecha_Apertura: {coincidencia_notif_apert:.2%}")
print(f"Fecha_Recepcion == Fecha_Apertura: {coincidencia_recep_apert:.2%}")
print(f"Las tres coinciden al mismo tiempo: {coincidencia_las_tres:.2%}")

In [0]:
meses = {1:'Enero',2:'Febrero',3:'Marzo',4:'Abril',5:'Mayo',6:'Junio',7:'Julio',
         8:'Agosto',9:'Septiembre',10:'Octubre',11:'Noviembre',12:'Diciembre'}

Base['mes_de_fecha_reporte'] = Base['Fecha de reporte'].dt.month.map(meses)

coincidencia_mes = (Base['mes_de_fecha_reporte'] == Base['Periodo Reporte']).mean()
coincidencia_anio = (Base['Fecha de reporte'].dt.year == Base['Año']).mean()

print(f"Mes de Fecha de reporte == Periodo Reporte: {coincidencia_mes:.2%}")
print(f"Año de Fecha de reporte == Año: {coincidencia_anio:.2%}")

Por último eliminamos las variables consideradas redundantes por definición, quedando con 29 de las 40 variables originales

In [0]:
columnas_redundantes = [
    'fecha_primera_vigencia_pol',   # se conserva Fecha_Primera_Vigencia_Cert
    'vigencia_poliza',              # se conserva vigencia_certificado
    'Fecha_Recepcion',              # se conserva F_Notificacion
    'Fecha_Apertura',               # se conserva F_Notificacion
    'Periodo Reporte',              # se conserva Fecha de reporte
    'Año'                           # se conserva Fecha de reporte
]

Base_sin_duplicados = Base_sin_duplicados.drop(columns=columnas_redundantes)

print(f"Columnas eliminadas: {columnas_redundantes}")
print(f"Dimensiones actuales: {Base_sin_duplicados.shape}")

#Revisión variables categóricas

In [0]:
columnas = [
    , 'Ramo_Desc', 'Nombre_plan', 'Codigo_Canal_Comercial_Op',
    'Nombre_Canal_Comercial', 'IDENTIFICACION_asegurado', 'SEXO_asegurado',
    'CODSUC', 'SUCURSAL', 'REGIONAL', 'AGENTE', 'CODAG', 'CAUSASTRO',
    'DESCAUSA', 'DIAGNOSTICO', 'Ind_Tipo_Atencion', 'Ind_Pago_Automatico',
    'estado', 'Cobertura', 'Tipo apertura', 'Fraude S /N', 'Periodo Reporte'
]

for col in columnas:
    valores = Base[col].unique()
    print(f"\n{'='*60}")
    print(f"Columna: {col}  |  Valores únicos: {len(valores)}")
    print(f"{'='*60}")
    if len(valores) <= 20:
        print(valores)
    else:
        print(f"(demasiados para mostrar todos, primeros 20): {valores[:20]}")


Con la ayuda de la función **MixedNulls()** de **deepchecks** podemos ver los **diferentes valores** que pueden ser interpredados como nulos , esto ayuda bastante para en el próximo paso podamos definir.

##Unificación de valores nulos
**Fecha_Primer_Cierre_Siniestro**: fecha del primer cierre. Ojo: hay registros con *1900-01-01*, que es un placeholder de "sin cerrar" y no una fecha real — hay que tratarlo como nulo.

segun se puede ver a simple vista esta base de datos representa un conjunto de reclamaciones de distinto tipo  hechas por clientes en diferentes ventanas de tiempo 

In [0]:
##Analisis de correlacion entre variables